In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys
import os

# Add the parent directory so we can import from the 'src' folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.eegnet import EEGNet
from src.data_prep import get_cleaned_epochs


In [ ]:
# Use the modular function we created to get the clean data
epochs, event_id = get_cleaned_epochs(subject_id=1)

# Convert MNE epochs to a NumPy array [Trials, Channels, TimePoints]
X = epochs.get_data() 
y = epochs.events[:, -1] - 769  # Normalize labels to start at 0


In [ ]:
import numpy as np

# 1. Get the raw event codes
raw_y = epochs.events[:, -1]

# 2. Identify the unique classes (e.g., 769, 770, 771, 772)
unique_classes = np.unique(raw_y)
print(f"Raw classes found in data: {unique_classes}")

# 3. Create a mapping to 0, 1, 2, 3
# This maps the smallest ID to 0, next to 1, etc.
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
print(f"Mapping labels to: {label_map}")

# 4. Apply the mapping
y = np.array([label_map[code] for code in raw_y])

# 5. Convert to Tensor
y_tensor = torch.tensor(y, dtype=torch.long)

# Now proceed to create your DataLoader...


In [ ]:
# Add the '1' dimension for the Convolutional layers
if X.ndim == 3:
    X = X[:, None, :, :]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create the DataLoader for batching
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)



In [ ]:
# 1. Initialize the model with dynamic dimensions
model = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 2. Setup the "Math" parts
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 3. The Training Loop
print("Starting training...")
for epoch in range(50):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Reset the math
        outputs = model(batch_X)        # Make a guess
        loss = criterion(outputs, batch_y) # Calculate the error
        loss.backward()                 # Calculate the fix
        optimizer.step()                # Apply the fix (Learning)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Training Complete!")
print(f"Model Flatten Size: {model.flatten_size}")
%load_ext autoreload
%autoreload 2
